# USGS Earthquake Dataset Header Dictionary

This schema represents standard earthquake event data provided by the **USGS (United States Geological Survey)**, annotated with recommendations for **time-series and spatiotemporal model training** (e.g., forecasting, sequence models, seismic hazard prediction).

## Data Summary & ML Training Suitability
| Header | Sample Value | Data Type | Unit / Interpretation | Description | Usable for Time-Series ML? |
|----------|----------|----------|----------|----------|----------|
| `time` | `2025-12-07T02:43:07.342Z` | ISO UTC Timestamp | Exact earthquake origin time in UTC. | Time when rupture began. | **Yes (Primary Time Index)** |
| `latitude` | `29.8698` | Float | 29.8698° North. Range: -90° to +90°. | North-South geographic position. | **Yes (Core Spatial Feature)** |
| `longitude` | `80.9384` | Float | 80.9384° East. Range: -180° to +180°. | East-West geographic position. | **Yes (Core Spatial Feature)** |
| `depth` | `43.404` | Float (km) | Earthquake started 43.404 km below the surface. | Hypocenter depth. | **Yes (Core Spatial Feature)** |
| `mag` | `4.4` | Float | Magnitude 4.4 earthquake. +1 magnitude ≈ 31.6× more energy. | Earthquake size/strength. | **Yes (Target Variable / Key Feature)** |
| `magType` | `mb` | String | Body-Wave Magnitude scale. Other values: `Mw`, `ML`, `Ms`. | Method used to compute magnitude. | **Optional (Categorical)** |
| `nst` | `21` | Integer | 21 seismic stations contributed to location estimation. | Station count used for location. | **Optional (Quality Weighting)** |
| `gap` | `166` | Float (degrees) | Largest angular gap between stations around the epicenter. Smaller is better. | Network coverage quality metric. | **Optional (Quality Weighting)** |
| `dmin` | `8.856` | Float (degrees) | Nearest station was ~983 km away (1° ≈ 111 km). | Distance to closest station. | **Optional (Quality Weighting)** |
| `rms` | `0.94` | Float (seconds) | Average travel-time prediction error of 0.94 s. Lower is better. | Model fit quality. | **Optional (Quality Weighting)** |
| `horizontalError` | `13.18` | Float (km) | Epicenter location uncertainty ≈ 13.18 km. | Horizontal location confidence. | **Optional (Uncertainty Weighting)** |
| `depthError` | `9.35` | Float (km) | Depth estimate uncertainty ≈ 9.35 km. | Depth confidence metric. | **Optional (Uncertainty Weighting)** |
| `magError` | `0.144` | Float | Magnitude uncertainty ±0.144. | Magnitude confidence metric. | **Optional (Uncertainty Weighting)** |
| `magNst` | `14` | Integer | 14 stations used specifically for magnitude calculation. | Magnitude measurement quality. | **Optional (Quality Weighting)** |
| `type` | `earthquake` | String | Event classification. | Earthquake, explosion, quarry blast, etc. | **Filter Only** |
| `status` | `reviewed` | String | Reviewed by a human seismologist. | Verification status. | **Filter Only** |
| `net` | `us` | String | USGS reporting network identifier. | Administrative metadata. | **No (Drop)** |
| `id` | `us7000rkd1` | String | Unique event identifier only. | Arbitrary database ID. | **No (Drop)** |
| `updated` | `2025-12-26T01:20:25.040Z` | ISO UTC Timestamp | Record modification timestamp. | Last update time. | **No (Drop — Leakage Risk)** |
| `place` | `37 km E of Dārchulā...` | String | Human-readable location description. | Textual location. | **No (Drop)** |
| `locationSource` | `us` | String | Agency that computed location. | Administrative metadata. | **No (Drop)** |
| `magSource` | `us` | String | Agency that computed magnitude. | Administrative metadata. | **No (Drop)** |

In [1]:
# CONSTANTS
ORGINAL_DATA_PATH = 'data/nepal_eq_90_26_raw.csv'
FILTERED_DATA_PATH = 'data/nepal_eq_90_26_filtered.csv'
INITIAL_EXTRACTION_DATA_PATH = 'data/nepal_eq_90_26_initial_extraction.csv'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df_original = pd.read_csv(ORGINAL_DATA_PATH)


# Filtering the dataset to include only relevant features
- Removing all the features that are not relevant to the analysis. The relevant features are mentioned 
in the Features class below. The filtered dataset is saved as 'data/nepal_eq_90_26_filtered.csv'.
and the rest columns are dropped since they are metadata about earthquate and the source of the data.

In [ ]:
from enum import Enum
class Features(Enum):
	TIME = 'time'
	LATITUDE = 'latitude'
	LONGITUDE = 'longitude'
	DEPTH = 'depth'
	MAGNITUDE = 'mag'
	MAGNITUDE_TYPE = 'magType'
	NSTATIONS = 'nst'
	GAP = 'gap'
	DMIN = 'dmin'
	RMS = 'rms'
	HORIZONTAL_ERROR = 'horizontalError'
	DEPTH_ERROR = 'depthError'
	MAGNITUDE_ERROR = 'magError'
	MAGNITUDE_NSTATIONS = 'magNst'	

	DISTRICT = 'district'

df_filtered = df_original[[f.value for f in Features if f != Features.DISTRICT]]
df_filtered.head()

df_filtered.to_csv(FILTERED_DATA_PATH, index=False)

In [ ]:
import numpy as np
from scipy.special import lambertw

def compute_felt_radius_with_depth(
	magnitude: float, 
	depth_km: float, 
	target_mmi: float = 2.5,
	region: str = "active"
) -> float:
	
	# Configuration Dictionary mapping different global tectonic regimes
	configurations = {
		"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
		"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
		"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US (High propagation)
		"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan, Chile megathrust
		"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California / Fractured active faults
		"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts (Very high attenuation)
	}
	
	if region in configurations:
		c1, c2, c3, c4 = configurations[region]
	else:
		raise ValueError(f"Unknown region profile: {region}")
	
	# 1. Saturation term
	
	R0 = 10 ** (-0.281 + 0.251 * magnitude)
	
	# 2. Lambert W setup for 3D effective distance R_eff
	a = c3 / c4
	b = (c1 + c2 * magnitude - target_mmi) / c4
	
	ln10 = np.log(10)
	z = (ln10 / a) * (10.0 ** (b / a))
	R_eff = (a / ln10) * lambertw(z).real
	
	# 3. Subtract saturation R0^2 AND focal depth h^2
	rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)
	
	if rad_sq <= 0:
		return 0.0  # Shaking absorbed before reaching surface at target_mmi
		
	return float(np.sqrt(rad_sq))


m = 7.8
h = 8.22
mmi = 2.5
region = "active"
r = compute_felt_radius_with_depth(magnitude=m, depth_km=h, target_mmi=mmi, region=region)

print(f"M{mmi} at {h}km depth with mag {m} in {region} region -> Felt Radius: {r:.1f} km")

M2.5 at 8.22km depth with mag 7.8 in active region -> Felt Radius: 833.3 km


In [21]:
import numpy as np
from scipy.special import lambertw

def compute_felt_radius_advanced(
    magnitude: float,
    depth_km: float,
    target_mmi: float = 2.5,
    region: str = "active",
    soil_type: str = "solid_rock"
) -> float:
    """
    Computes surface felt radius R_felt (km): the epicentral distance at which
    ground shaking is predicted to reach `target_mmi`, combining a regional
    tectonic attenuation model with a local soil/rock (Vs30) amplification term.
    """
    # 1. Master Configuration Dictionaries
    coefficient_configurations = {
        "active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
        "stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
        "stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US
        "subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan megathrust
        "strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California faults
        "volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts
    }

    soil_configurations = {
        "hard_rock":   (1500.0, "Hard bedrock"),
        "solid_rock":  (760.0,  "Standard baseline engineering rock"),
        "soft_rock":   (560.0,  "Dense soil / weathered rock"),
        "stiff_soil":  (270.0,  "Stiff urban soil"),
        "soft_basin":  (150.0,  "Loose alluvium / deep basin sediments"),
        "swamp_mud":   (100.0,  "Ultra-soft marsh muds")
    }

    # 2. Extract settings with safety defaults
    c1, c2, c3, c4 = coefficient_configurations.get(region, coefficient_configurations["active"])
    vs30, _ = soil_configurations.get(soil_type, soil_configurations["solid_rock"])

    # 3. Site amplification relative to 760 m/s baseline rock
    v_ref = 760.0
    b_site = 1.8
    delta_mmi_site = b_site * np.log10(v_ref / vs30)

    # Back out the site term from the target: raises the required base-rock
    # intensity on hard rock (delta<0), lowers it on soft soil (delta>0)
    adjusted_target_mmi = target_mmi - delta_mmi_site

    # 4. Finite-fault saturation term
    R0 = 10 ** (-0.281 + 0.251 * magnitude)

    # 5. Lambert W setup for 3D effective distance
    a = c3 / c4
    b = (c1 + c2 * magnitude - adjusted_target_mmi) / c4

    ln10 = np.log(10)
    z = (ln10 / a) * (10.0 ** (b / a))
    R_eff = (a / ln10) * lambertw(z).real

    # 6. Pythagorean depth / finite-fault correction to surface distance
    rad_sq = R_eff**2 - R0**2 - (depth_km ** 2)

    if rad_sq <= 0:
        return 0.0

    return float(np.sqrt(rad_sq))

# --- Executing a Multivariable Grid Test ---
m, h, target = 7.8, 8.22, 3

print(f"--- MMI\u2265{target} Felt Radius Boundaries for a Mag {m} Event (Depth: {h}km) ---")
for soil in ["hard_rock", "solid_rock", "stiff_soil", "soft_basin"]:
    radius = compute_felt_radius_advanced(m, h, target_mmi=target, region="active", soil_type=soil)
    print(f"-> Region: active | Rock Profile: {soil:<12} -> Radius: {radius:.1f} km")

--- MMI≥3 Felt Radius Boundaries for a Mag 7.8 Event (Depth: 8.22km) ---
-> Region: active | Rock Profile: hard_rock    -> Radius: 516.7 km
-> Region: active | Rock Profile: solid_rock   -> Radius: 672.1 km
-> Region: active | Rock Profile: stiff_soil   -> Radius: 939.2 km
-> Region: active | Rock Profile: soft_basin   -> Radius: 1104.1 km


In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.special import lambertw


@dataclass(frozen=True)
class FeltRadiusConfig:
	"""
	Configures the parameters for felt radius calculation.
	Allows customization of input and output column names, as well as the region type.
	Note: i_ = input column, o_ = output column
	"""
	i_region: str = "active"  # "active" or "stable"
	i_time_column: str = "time"
	i_mag_column: str = "mag"
	i_depth_column: str = "depth"
	o_felt_radius_column: str = "felt_radius_km"
	o_felt_area_column: str = "felt_area_sqkm"
	o_max_nearfield_mmi_column: str = "max_nearfield_mmi"


class FeltRadiusCalculator:
	coefficient_configurations = {
	"active":      (2.08, 1.04, 2.09, 0.0019),   # e.g., Himalayas / Alpine Belt
	"stable":      (1.85, 1.15, 1.80, 0.0006),   # Standard stable continental crust
	"stable_cena": (1.35, 1.21, 1.70, 0.00035),  # Central & Eastern US (High propagation)
	"subduction":  (2.45, 0.98, 2.15, 0.0012),   # Cascadia, Japan, Chile megathrust
	"strike_slip": (1.95, 1.05, 2.20, 0.0025),   # California / Fractured active faults
	"volcanic":    (2.10, 1.02, 2.35, 0.0041)    # Iceland / Rifts (Very high attenuation)
	}

	@staticmethod
	def extract_seismic_features(
		df: pd.DataFrame, 
		felt_mmi: float = 2.5, 
		config: FeltRadiusConfig = FeltRadiusConfig()
	) -> pd.DataFrame:
		"""
		Extracts physical felt-vibration features from earthquake catalog data.
		Preserves original row order and input column types.
		"""
		data = df.copy()
		coefficients = FeltRadiusCalculator.coefficient_configurations
		if region in FeltRadiusCalculator.coefficient_configurations:
				c1, c2, c3, c4 = coefficients[region]
		else:
			raise ValueError(f"Unknown region profile: {region}")
			
		M = data[config.i_mag_column].values
		h = data[config.i_depth_column].fillna(10.0).values  # Default to 10km depth if missing
		
		# 1. Near-field saturation radius R0(M)
		R0 = 10 ** (-0.281 + 0.251 * M)
		
		# 2. Transformed Lambert W parameters (a = c3/c4, b = (c1 + c2*M - MMI)/c4)
		a = c3 / c4
		b = (c1 + c2 * M - felt_mmi) / c4
		
		# 3. Vectorized Lambert W evaluation for R_eff
		ln10 = np.log(10)
		z = (ln10 / a) * (10.0 ** (b / a))
		R_eff = (a / ln10) * lambertw(z).real
		
		# 4. Surface felt radius accounting for depth (h) and saturation (R0)
		# Prevent RuntimeWarning by clipping negative rad_sq values to 0.0 before np.sqrt
		rad_sq = R_eff**2 - R0**2 - h**2
		r_felt = np.sqrt(np.maximum(rad_sq, 0.0))
		
		# 5. Append new physical feature columns without altering existing input columns
		data[config.o_felt_radius_column] = r_felt
		data[config.o_felt_area_column] = np.pi * (r_felt ** 2)
		
		# 6. Peak intensity directly above hypocenter (R_jb = 0)
		R_eff_epicenter = np.sqrt(R0**2 + h**2)
		data[config.o_max_nearfield_mmi_column] = (
			c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
		)
		return data


# Execution
config = FeltRadiusConfig(
	i_region="active",
	i_time_column="time",
	i_mag_column="mag",
	i_depth_column="depth",
	o_felt_radius_column="felt_radius_km",
	o_felt_area_column="felt_area_sqkm",
	o_max_nearfield_mmi_column="max_nearfield_mmi"
)

mmi = 4.0
data_with_felt_features = FeltRadiusCalculator.extract_seismic_features(
	df_filtered, 
	felt_mmi=mmi, 
	config=config
)
data_with_felt_features.to_csv(INITIAL_EXTRACTION_DATA_PATH, index=False)

In [ ]:
# print(data_with_felt_features.describe())
minimal = data_with_felt_features["felt_radius_km"].sort_values(ascending=False)[:5]
print(data_with_felt_features.loc[minimal.index])


                         time  latitude  longitude  depth  mag magType    nst  \
704  2015-04-25T06:11:25.950Z   28.2305    84.7314   8.22  7.8     mww    NaN   
545  2015-05-12T07:05:19.730Z   27.8087    86.0655  15.00  7.3     mww    NaN   
122  2025-01-07T01:05:16.734Z   28.6036    87.3781  10.00  7.1     mww  315.0   
778  2011-09-18T12:40:51.830Z   27.7300    88.1550  50.00  6.9     mww  383.0   
961  2008-08-25T13:21:58.820Z   30.9010    83.5200  12.00  6.7     mwc  325.0   

      gap   dmin   rms  horizontalError  depthError  magError  magNst  \
704  12.0  1.857  0.96             7.30        2.90       NaN     NaN   
545  11.0  0.677  0.92             7.40        1.70       NaN     NaN   
122  17.0  3.458  0.56             8.15        1.74     0.039    63.0   
778  13.0    NaN  0.96              NaN         NaN       NaN     NaN   
961  28.0    NaN  1.07              NaN         NaN       NaN     NaN   

     felt_radius_km  felt_area_sqkm  max_nearfield_mmi  
704      395.5612

In [11]:

minimal = data_with_felt_features["felt_radius_km"].sort_values(ascending=True)[:5]
print(data_with_felt_features.loc[minimal.index])


                          time  latitude  longitude  depth  mag magType   nst  \
892   2008-09-04T20:55:44.500Z    30.739     83.187    6.4  3.6      mb  17.0   
1383  1996-08-03T07:12:06.420Z    30.076     88.255   33.0  4.1      mb   NaN   
1275  1998-10-02T17:36:44.620Z    29.170     85.136   33.0  4.2      mb   NaN   
1277  1998-09-26T20:48:44.110Z    27.521     86.135   33.0  3.9      mb   NaN   
1278  1998-09-21T23:24:40.670Z    29.940     88.005   33.0  4.1      mb   NaN   

        gap  dmin   rms  horizontalError  depthError  magError  magNst  \
892   145.2   NaN  1.33              NaN        43.1       NaN     2.0   
1383    NaN   NaN  0.50              NaN         NaN       NaN    10.0   
1275    NaN   NaN  1.12              NaN         NaN       NaN     3.0   
1277    NaN   NaN  1.13              NaN         NaN       NaN     1.0   
1278    NaN   NaN  0.97              NaN         NaN       NaN     3.0   

      felt_radius_km  felt_area_sqkm  max_nearfield_mmi  
892       

In [ ]:
# import numpy as np
# import plotly.graph_objects as go

# def generate_circle_perimeter(lat: float, lon: float, radius_km: float, num_points: int = 40):
#     """Calculates geodesic perimeter lat/lon coordinates on a sphere."""
#     if radius_km <= 0:
#         return [], []
#     R = 6371.0  # Earth radius in km
#     angles = np.linspace(0, 2 * np.pi, num_points)
#     lat_rad, lon_rad = np.radians(lat), np.radians(lon)
#     d_R = radius_km / R

#     circle_lats = np.arcsin(
#         np.sin(lat_rad) * np.cos(d_R) + 
#         np.cos(lat_rad) * np.sin(d_R) * np.cos(angles)
#     )
#     circle_lons = lon_rad + np.arctan2(
#         np.sin(angles) * np.sin(d_R) * np.cos(lat_rad),
#         np.cos(d_R) - np.sin(lat_rad) * np.sin(circle_lats)
#     )
#     return np.degrees(circle_lats), np.degrees(circle_lons)


# fig = go.Figure()

# # 1. Epicenter points using go.Scattermap
# fig.add_trace(go.Scattermap(
#     lat=data_with_felt_features["latitude"],
#     lon=data_with_felt_features["longitude"],
#     mode="markers",
#     marker=dict(size=7, color="black"),
#     name="Epicenters",
#     text=data_with_felt_features["mag"],
#     hovertemplate="<b>Mag: %{text}</b><br>Lat: %{lat:.3f}<br>Lon: %{lon:.3f}<extra></extra>"
# ))

# # 2. True-scale physical circles
# for _, row in data_with_felt_features.iterrows():
#     r_km = row["felt_radius_km"]
#     if r_km > 0:
#         c_lats, c_lons = generate_circle_perimeter(row["latitude"], row["longitude"], r_km)
		
#         fig.add_trace(go.Scattermap(
#             lat=c_lats,
#             lon=c_lons,
#             mode="lines",
#             fill="toself",
#             fillcolor="rgba(239, 68, 68, 0.20)",
#             line=dict(color="darkred", width=1.5),
#             showlegend=False,
#             hoverinfo="skip"
#         ))

# # 3. Layout uses map= instead of mapbox=
# fig.update_layout(
#     map=dict(
#         style="open-street-map",  # Options: 'open-street-map', 'carto-positron', 'carto-darkmatter'
#         center=dict(lat=28.3949, lon=84.1240),
#         zoom=6.2,
#     ),
#     margin={"r": 0, "t": 40, "l": 0, "b": 0},
#     title="Nepal Earthquakes — Geodesic Felt Extent (km) Overlay"
# )

# fig.show()